In [224]:
# Standard Headers
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
import numpy as np

In [225]:
# Read in the data
train_data = pd.read_csv("data/train.csv", skipinitialspace=True, low_memory=False, na_values = ["UNKNOWN", "Unknown", "UNK"])

# What percent of data values are null
print("Percent Null:", train_data.isnull().sum().sum() / (len(train_data) * len(train_data.columns)) * 100)

# Null amount for each column
print(train_data.isnull().sum())

Percent Null: 30.561646519429953
INDEX_NR                     0
INCIDENT_DATE                0
INCIDENT_MONTH               0
INCIDENT_YEAR                0
TIME                    132042
TIME_OF_DAY             133971
AIRPORT_ID                   0
AIRPORT                  40684
LATITUDE                 40744
LONGITUDE                40747
RUNWAY                   75586
STATE                    40744
FAAREGION                40744
LOCATION                267965
OPID                     86692
OPERATOR                 86692
REG                     118999
FLT                     161442
AIRCRAFT                 86942
AMA                      88896
AMO                     116813
EMA                     102421
EMO                     113176
AC_CLASS                 87315
AC_MASS                  87446
TYPE_ENG                 87786
NUM_ENGS                 87697
ENG_1_POS                87717
ENG_2_POS               102509
ENG_3_POS               294822
ENG_4_POS               303909
PHASE_

For each column, drop the column if it has 50% or more of its data missing

In [226]:
# For each column, drop the column if it has 50% or more of its data missing
print("Before:", len(train_data.columns))

for col in train_data:
    if train_data[col].isnull().sum() >= len(train_data) / 2:
        train_data.drop(col, inplace=True, axis=1)
        print(col)

print("After:", len(train_data.columns))

Before: 55
LOCATION
FLT
ENG_3_POS
ENG_4_POS
HEIGHT
SPEED
SKY
PRECIPITATION
BIRD_BAND_NUMBER
WARNED
NUM_SEEN
ENROUTE_STATE
After: 43


Drop highly correlated columns

In [227]:
print("Before:", len(train_data.columns))
train_data.drop("AIRPORT", inplace=True, axis=1)
train_data.drop("OPERATOR", inplace=True, axis=1)
train_data.drop("SPECIES", inplace=True, axis=1)
print("After:", len(train_data.columns))

Before: 43
After: 40


Drop columns with no usefullness

In [228]:
print("Before:", len(train_data.columns))
train_data.drop("REMARKS", inplace=True, axis=1)
train_data.drop("REMAINS_SENT", inplace=True, axis=1)
train_data.drop("COMMENTS", inplace=True, axis=1)
train_data.drop("SOURCE", inplace=True, axis=1)
train_data.drop("PERSON", inplace=True, axis=1)
train_data.drop("LUPDATE", inplace=True, axis=1)
train_data.drop("TRANSFER", inplace=True, axis=1)
train_data.drop("NUM_STRUCK", inplace=True, axis=1)
train_data.drop("INDEX_NR", inplace=True, axis=1) # Eliminate since it is an extra index
print("After:", len(train_data.columns))

Before: 40
After: 31


Dealing with remaining null values.

In [229]:
print(train_data.isnull().sum()[train_data.isnull().sum() > 0] / len(train_data) * 100)

TIME               42.985500
TIME_OF_DAY        43.613475
LATITUDE           13.263971
LONGITUDE          13.264947
RUNWAY             24.606580
STATE              13.263971
FAAREGION          13.263971
OPID               28.222073
REG                38.739428
AIRCRAFT           28.303459
AMA                28.939572
AMO                38.027788
EMA                33.342557
EMO                36.843784
AC_CLASS           28.424887
AC_MASS            28.467533
TYPE_ENG           28.578218
NUM_ENGS           28.549245
ENG_1_POS          28.555756
ENG_2_POS          33.371205
PHASE_OF_FLIGHT    39.381075
DISTANCE           33.629687
SIZE               10.908008
dtype: float64


Filling missing numeric values with median of that column

In [230]:
print("Before:", len(train_data.columns))

# Find the comma errors where latitude and longitude were combined
comma_errors = train_data["LATITUDE"].str.contains(",", na=False)
vals_with_com_errs = train_data.loc[comma_errors, "LATITUDE"].str.split(",", expand=True)

# Fix the comma errors
train_data.loc[comma_errors, "LATITUDE"] = vals_with_com_errs[0]
train_data.loc[comma_errors, "LONGITUDE"] = vals_with_com_errs[1]

# Make LAT and LONG numeric floats
train_data["LATITUDE"] = pd.to_numeric(train_data["LATITUDE"], errors="coerce")
train_data["LONGITUDE"] = pd.to_numeric(train_data["LONGITUDE"], errors="coerce")

# Fill in the missing values for LAT and LONG, drop AIRPORT_ID and STATE
for col in ["LATITUDE", "LONGITUDE"]:
    train_data[col] = train_data[col].fillna(
        train_data.groupby("AIRPORT_ID")[col].transform("median")
    )

    train_data[col] = train_data[col].fillna(
        train_data.groupby("STATE")[col].transform("median")
    )

    train_data[col] = train_data[col].fillna(train_data[col].median())
train_data.drop("AIRPORT_ID", inplace=True, axis=1)
train_data.drop("STATE", inplace=True, axis=1)

# Extracting out the hour value from the time column
train_data["TIME"] = pd.to_datetime(train_data["TIME"], format="%H:%M", errors="coerce").dt.round("h").dt.hour

# Filling missing TIME values with the median value of it's TIME_OF_DAY
train_data["TIME"] = train_data.groupby("TIME_OF_DAY")["TIME"].transform(
    lambda x: x.fillna(x.median())
).round().astype("Int64")


# Filling rest of missing TIME values with median time
train_data["TIME"] = train_data["TIME"].fillna(train_data["TIME"].median())


# Drop TIME_OF_DAY column  
train_data.drop("TIME_OF_DAY", inplace=True, axis=1)

# Use PHASE_OF_FLIGHT to fill in nans for DISTANCE
train_data["DISTANCE"] = train_data["DISTANCE"].fillna(
    train_data.groupby("PHASE_OF_FLIGHT")["DISTANCE"].transform("median")
)

# Just use median DISTANCE for nans if PHASE_OF_FLIGHT is also missing for this data point
train_data["DISTANCE"] = train_data["DISTANCE"].fillna(train_data["DISTANCE"].median())

print("After:", len(train_data.columns))
print(train_data.isnull().sum() / len(train_data) * 100)

Before: 31
After: 28
INCIDENT_DATE            0.000000
INCIDENT_MONTH           0.000000
INCIDENT_YEAR            0.000000
TIME                     0.000000
LATITUDE                 0.000000
LONGITUDE                0.000000
RUNWAY                  24.606580
FAAREGION               13.263971
OPID                    28.222073
REG                     38.739428
AIRCRAFT                28.303459
AMA                     28.939572
AMO                     38.027788
EMA                     33.342557
EMO                     36.843784
AC_CLASS                28.424887
AC_MASS                 28.467533
TYPE_ENG                28.578218
NUM_ENGS                28.549245
ENG_1_POS               28.555756
ENG_2_POS               33.371205
PHASE_OF_FLIGHT         39.381075
DISTANCE                 0.000000
SPECIES_ID               0.000000
OUT_OF_RANGE_SPECIES     0.000000
REMAINS_COLLECTED        0.000000
SIZE                    10.908008
INDICATED_DAMAGE         0.000000
dtype: float64
